# STAT 207 Group Lab Assignment 14 - [10 total points]

## Regularization Models & Selecting a Tuning Parameter

<hr>

## <u>Lab Grading</u>:

Should we grade your submission?  If not, write the netID of the submission to be graded.  (Note: We will only grade one assignment per group, and we'll pick the first one that says we should grade that submission.  We will assign the same grade to all team members.)

*For example*, you might respond: **grade this submission** or **my submission is under netID jdeeke**

my submission is under netID rjguhl2

If you said **my submission is under netID** above, we will not read any more of your lab submission.

If you said **grade this submission** above, who worked with you on this submission?  Write both their **names** and **netIDs**.  

Also, discuss and record if you've ever been on a cruise before.

I have never been on a cruise

## <u>Purpose</u>:
You should work in groups of 2-3 on this report (not working in groups without permission will result in a point deduction). The purpose of this group lab assignment is to fit a regularized model and assess its performance on new data.
<hr>

## <u>Assignment Instructions</u>:

### Group Roles

Suggested and specified roles are provided below: 

#### Groups of 2

* **Driver**: This student will type the report.  While typing the report, you may be the one who is selecting the functions to apply to the data.
* **Navigator**: This student will guide the process of answering the question.  Specific ways to help may include: outlining the general steps needed to solve a question (providing the overview), locating examples within the course notes, and reviewing each line of code as it is typed.

#### Groups of 3

* **Driver**: This student will type the report.  They may also be the one to select the functions to apply to the data.
* **Navigator**: This student will guide the process of answering the question.  They may select the general approach to answering the question and/or a few steps to be completed along the way. 
* **Communicator**: This student will review the report (as it is typed) to ensure that it is clear and concise.  This student may also locate relevant examples within the course notes that may help complete the assignment.

<hr>

### Imports

In [88]:
#Run this
import pandas as pd                    # imports pandas and calls the imported version 'pd'
import matplotlib.pyplot as plt        # imports the package and calls it 'plt'
import seaborn as sns                  # imports the seaborn package with the imported name 'sns'
sns.set()  

## Case Study: Avoiding Underwater Weighing

We will look at data collected on 252 males in 1985.  In particular, we will consider a measure of the percent of body fat in these males, as measured by Siri's equation (`siri`).  This particular measure is obtained through an underwater weighing technique and is quite extensive and resource-intensive.  We would like to consider alternative (and easier to capture information) to get roughly the same information that the `siri` variable currently contains.  We have other body measures for these males available, including:

- **age**: Age (yrs)
- **weight**: Weight (lbs)
- **height**: Height (inches)
- **adipos**: BMI index
- **neck**: Neck circumference (cm)
- **chest**: Chest circumference (cm)
- **abdom**: Abdomen circumference (cm)
- **hip**: Hip circumference (cm)
- **dthigh**: Thigh circumference (cm)
- **knee**: Knee circumference (cm)
- **ankle**: Ankle circumference (cm)
- **biceps**: Extended bicepts circumference (cm)
- **forearm**: Forearm circumference (cm)
- **wrist**: Wrist circumference (cm)

**We will use all variables in the data as predictors except for the siri, brozek, density, and free (fat free weight)**, since these four variables are challenging measurements to obtain, for our analysis.

The code cell below will read in the data for you.  Be sure to run the cell. 

In [89]:
df = pd.read_csv('fat.csv')
df

,brozek,siri,density,age,weight,height,adipos,free,neck,chest,abdom,hip,thigh,knee,ankle,biceps,forearm,wrist
0,12.6,12.3,1.0708,23,154.25,67.75,23.7,134.9,36.2,93.1,85.2,94.5,59.0,37.3,21.9,32.0,27.4,17.1
1,6.9,6.1,1.0853,22,173.25,72.25,23.4,161.3,38.5,93.6,83.0,98.7,58.7,37.3,23.4,30.5,28.9,18.2
2,24.6,25.3,1.0414,22,154.00,66.25,24.7,116.0,34.0,95.8,87.9,99.2,59.6,38.9,24.0,28.8,25.2,16.6
3,10.9,10.4,1.0751,26,184.75,72.25,24.9,164.7,37.4,101.8,86.4,101.2,60.1,37.3,22.8,32.4,29.4,18.2
4,27.8,28.7,1.0340,24,184.25,71.25,25.6,133.1,34.4,97.3,100.0,101.9,63.2,42.2,24.0,32.2,27.7,17.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,11.5,11.0,1.0736,70,134.25,67.00,21.1,118.9,34.9,89.2,83.6,88.8,49.6,34.8,21.5,25.6,25.7,18.5
248,32.3,33.6,1.0236,72,201.00,69.75,29.1,136.1,40.9,108.5,105.0,104.5,59.6,40.8,23.2,35.2,28.6,20.1
249,28.3,29.3,1.0328,72,186.75,66.00,30.2,133.9,38.9,111.1,111.5,101.7,60.3,37.3,21.5,31.3,27.2,18.0
250,25.3,26.0,1.0399,72,190.75,70.50,27.0,142.6,38.9,108.3,101.3,97.8,56.0,41.6,22.7,30.5,29.4,19.8


### 1. [1 point] Select a Regularization Technique 

We work for a company that is measuring the proportion of body fat for customers in order to design suits that can fit well for a customer that is ordering the suit over the internet.  We know that asking these customers to perform many body measurements will be burdensome for the customers, so we'd like to streamline the process by only asking the customer to provide a few measurements.  We can then use our models to select an appropriately cut and tailored suit for the customer.

**a)** For this situation, what would our primary purpose of fitting the model be?  In other words, are we concerned about making predictions or understanding structures?

Our goal is prediction, not understanding structures. To recommend a tailored suit, we want to predict a customer’s body fat percentage based on a few easy-to-measure body dimensions.

**b)** We would like to use a regularization model for our fitted model.  What regularization technique would you suggest should be used based on the optimal design for the company and customer?

We would use Lasso because it performs feature selection by shrinking some coefficients to zero, helping us identify the most relevant body measurements while ignoring the rest as it simplifies the model and reduces customer burden by minimizing the number of measurements required.

### 2. [2.5 points] Prepare the Data

**a)** Split your data into training and test data.  Use a random state (you can choose your random state), and set aside 20% of your data for the test data.

In [90]:
from sklearn.model_selection import train_test_split

X = df.drop(['siri'], axis=1)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

**b)** We know that regularization models require some preparation of the data before the model can be fit.  Scale your $X$ explanatory variables for this model.

*Note:* You should perform this scaling in two stages.  You should fit a scaler and then scale your training data.  For your test data, you should use the scaler trained by your training data and apply it to your test data.

In [91]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) 

**c)** Prepare your y variable for the model.

In [92]:
y = df['siri']
y_train, y_test = train_test_split(y, test_size=0.2, random_state=42)

### 3. [3 points] Fitting a Regularized Model

**a)** We'd like to fit a model using the regularization technique suggested in Question 1.  Start by fitting your model with a $\lambda$ of 2 to your training data.

In [93]:
from sklearn.linear_model import Lasso

lasso_2 = Lasso(alpha=2)
lasso_2.fit(X_train_scaled, y_train)
coef_2 = lasso_2.coef_
coef_2

array([ 6.67239981, -0.        ,  0.        ,  0.        , -0.        ,
        0.        , -0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ])

**b)** How many of your slopes were equal to 0 for this model?  Which variables would you suggest should be retained in the model?

16. Brozek

In [94]:
coef_2 = lasso_2.coef_
retained_vars_2 = X.columns[coef_2 != 0]
print(f"Number of slopes = 0: {sum(coef_2 == 0)}")
print(f"Retained variables: {list(retained_vars_2)}")

Number of slopes = 0: 16
Retained variables: ['brozek']


**c)** Fit another version of your regularized model to your training data, but this time using a $\lambda$ of 0.5.

In [95]:
from sklearn.linear_model import Lasso

lasso_05 = Lasso(alpha=0.5)
lasso_05.fit(X_train_scaled, y_train)
coef_05 = lasso_05.coef_
coef_05

array([ 8.17239981, -0.        ,  0.        ,  0.        , -0.        ,
        0.        , -0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ])

**d)** How many of your slopes were equal to 0 for this model?  Which variables would you suggest should be retained from this version of the model?

In [96]:
coef_05 = lasso_05.coef_
retained_vars_05 = X.columns[coef_05 != 0]
print(f"Number of slopes = 0: {sum(coef_05 == 0)}")
print(f"Retained variables: {list(retained_vars_05)}")

Number of slopes = 0: 16
Retained variables: ['brozek']


16 Brozek

### 4. [2 points] Comparing Model Results

**a)** Apply each of your two models fit from **3a** and **3c** to your test set.  Calculate the $R^2$ on your test set for each of your models.

In [97]:
from sklearn.metrics import r2_score

y_pred_2 = lasso_2.predict(X_test_scaled)
r2_2 = r2_score(y_test, y_pred_2)
r2_2

0.9439761501341991

In [98]:
y_pred_05 = lasso_05.predict(X_test_scaled)
r2_05 = r2_score(y_test, y_pred_05)
r2_05

0.9962330426551432

**b)** Which $\lambda$ would you suggest based on your training data?

Choose the λ that provides the better R^2 on the test data. If λ = 0.5 gives better results, it indicates a more flexible model that fits the data better.

**c)** Compare the results from **4a** to another group (or two) in your lab.  Did you pick the same preferred $\lambda$ from our two choices?  Did you achieve the same $R^2$ values on your test data?  Did you have the same training/test data split from **2a**?

Same conclusions but different values for R^2

### 5. [1.5 points] A Limitation of the Results

Return to your selection of a regularization technique in Question 1 and recall that there are two primary purposes or benefits to using a regularization model.  Assess if there is still a concern in your model that the other regularization model approach could address.  Explain.

While Lasso simplifies the model by selecting relevant variables, it might drop some important variables or over-simplify the model. Ridge regression retains all variables and avoids this risk.